#Telecom Domain ReadOps Assignment
This notebook contains assignments to practice Spark read options and Databricks volumes. <br>
Sections: Sample data creation, Catalog & Volume creation, Copying data into Volumes, Path glob/recursive reads, toDF() column renaming variants, inferSchema/header/separator experiments, and exercises.<br>

![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

##First Import all required libraries & Create spark session object

In [0]:
from pyspark.sql.session import SparkSession
print(spark)#default databricks session instantiated
spark1 = SparkSession.builder.getOrCreate()
print(spark1)#user instantiated spark object, both refers to same object

##1. Write SQL statements to create:
1. A catalog named telecom_catalog_assign
2. A schema landing_zone
3. A volume landing_vol
4. Using dbutils.fs.mkdirs, create folders:<br>
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/
5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):<br>
a. Volume vs DBFS/FileStore<br>
b. Why production teams prefer Volumes for regulated data<br>

In [0]:
%sql
create catalog if not exists telecom_catalog_assign;
create database if not exists telecom_catalog_assign.landing_zone;
create volume if not exists telecom_catalog_assign.landing_zone.landing_vol;

In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/")

##Data files to use in this usecase:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

# **Data files to use in this usecase:**
customer_csv = ''' 101,Arun,31,Chennai,PREPAID <br>
102,Meera,45,Bangalore,POSTPAID <br>
103,Irfan,29,Hyderabad,PREPAID <br>
104,Raj,52,Mumbai,POSTPAID <br>
105,,27,Delhi,PREPAID <br>
106,Sneha,abc,Pune,PREPAID '''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count <br>
101\t320\t1500\t20 <br>
102\t120\t4000\t5 <br>
103\t540\t600\t52 <br>
104\t45\t200\t2 <br>
105\t0\t0\t0 '''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp <br>
5001|101|TWR01|-80|2025-01-10 10:21:54 <br>
5004|104|TWR05|-75|2025-01-10 11:01:12 '''

##2. Filesystem operations
1. Write code to copy the above datasets into your created Volume folders:
Customer → /Volumes/.../customer/
Usage → /Volumes/.../usage/
Tower (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/

2. Write a command to validate whether files were successfully copied

In [0]:
customer_csv = """ 101,Arun,31,Chennai,PREPAID 
102,Meera,45,Bangalore,POSTPAID 
103,Irfan,29,Hyderabad,PREPAID 
104,Raj,52,Mumbai,POSTPAID 
105,,27,Delhi,PREPAID 
106,Sneha,abc,Pune,PREPAID """

usage_tsv = """customer_id\tvoice_mins\tdata_mb\tsms_count 
101\t320\t1500\t20 
102\t120\t4000\t5 
103\t540\t600\t52 
104\t45\t200\t2 
105\t0\t0\t0 """

tower_logs_region1 = """event_id|customer_id|tower_id|signal_strength|timestamp 
5001|101|TWR01|-80|2025-01-10 10:21:54 
5004|104|TWR05|-75|2025-01-10 11:01:12 """

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv", customer_csv, True)
dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.csv", usage_tsv, True)
dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_logs_region1", tower_logs_region1, True)

In [0]:
#move file from tower to region1 dir
dbutils.fs.cp("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/",True)

##3. Directory Read Use Cases
1. Read all tower logs using:
Path glob filter (example: *.csv)
Multiple paths input
Recursive lookup

2. Demonstrate these 3 reads separately:
Using pathGlobFilter
Using list of paths in spark.read.csv([path1, path2])
Using .option("recursiveFileLookup","true")

3. Compare the outputs and understand when each should be used.

In [0]:
#recursiveFileLookup=True, reads files from the subfolders too
#pathGlobFilter="tower_logs_*", reads files with the pattern starts with tower_logs_, if file name unknown, we can use *.csv
df_multiple_path_files=spark.read.csv(path=["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2"],header=True,inferSchema=True,sep="|",recursiveFileLookup=True,pathGlobFilter="tower_logs_*")
print(df_multiple_path_files.count())


##4. Schema Inference, Header, and Separator
1. Try the Customer, Usage files with the option and options using read.csv and format function:<br>
header=false, inferSchema=false<br>
or<br>
header=true, inferSchema=true<br>
2. Write a note on What changed when we use header or inferSchema  with true/false?<br>
3. How schema inference handled “abc” in age?<br>

In [0]:
'''Try the Customer, Usage files with the option and options using read.csv and format function:
header=false, inferSchema=false
or
header=true, inferSchema=true'''
df1 = spark1.read.options(header="false", inferSchema="false").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv")
df1.printSchema
display(df1)

- while using header=True, its taking 1st row as a column name, in case of False, its taking c0,c1,c2, so on by default
- if we use toDF with header=False, it will take user specified column name instead of default
- if we use toDF with header=True, it will take user specified column name instead of default but 1st row is getting vanished - not recommended (leads to data loss)
- inferSchema=True - based on data, it will assign the column datatype
- inferSchema=False - by default string is a datatype

### output
> Age datatype considered as String as age of Sneha mentioned as "abc", if not str else number, datatype will be an integer. <br>
> Null is present in the custname.


##5. Column Renaming Usecases
1. Apply column names using string using toDF function for customer data
2. Apply column names and datatype using the schema function for usage data
3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data 

In [0]:
#1. Apply column names using string using toDF function for customer data
df1 = spark.read.options(inferSchema="true").format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv").toDF("customer_id","name","age","city","plan")
display(df1)

#2. Apply column names and datatype using the schema function for usage data
str_struct="customer_id integer, voice_mins integer, data_mb integer, sms_count string"
use_schema=spark.read.schema(str_struct).options(header=True,sep="\t").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.csv")
display(use_schema)

#3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
cust_schema=StructType(
    [
        StructField("event_id",IntegerType(),True),
        StructField("customer_id",IntegerType(),True),
        StructField("tower_id",StringType(),True),
        StructField("signal_strength",StringType(),True),
        StructField("timestamp",StringType(),True)
    ]
)
df_cust_schema=spark.read.schema(cust_schema).options(header=True,sep="|").format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_region1")
display(df_cust_schema)

## Spark Write Operations using 
- csv, json, orc, parquet, delta, saveAsTable, insertInto, xml with different write mode, header and sep options

##6. Write Operations (Data Conversion/Schema migration) – CSV Format Usecases
1. Write customer data into CSV format using overwrite mode
2. Write usage data into CSV format using append mode
3. Write tower data into CSV format with header enabled and custom separator (|)
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:
#read data
rd_df=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",inferSchema=True).toDF("custid","name","age","location","plan")

#1. Write customer data into CSV format using overwrite mode
wr_df=rd_df.write.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer",mode="Overwrite")

In [0]:
#2. Write usage data into CSV format using append mode
rd_df1=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_updated.csv",header=True,inferSchema=True)
rd_df1.write.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/usage",header=True,mode="append")

#3. Write tower data into CSV format with header enabled and custom separator (|)
rd_df2=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_region1",header=True,sep="|")
wr_tower=rd_df2.write.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/tower",header=True,sep="|",mode="Overwrite")

#4. Read the tower data in a dataframe and show only 5 rows.
rd_df3=spark.read.option("header","true").option("sep","|").csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/tower",pathGlobFilter="*.csv").limit(5)
display(rd_df3)


##7. Write Operations (Data Conversion/Schema migration)– JSON Format Usecases
1. Write customer data into JSON format using overwrite mode
2. Write usage data into JSON format using append mode and snappy compression format
3. Write tower data into JSON format using ignore mode and observe the behavior of this mode
4. Read the tower data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:
#1. Write customer data into JSON format using overwrite mode
wr_json1=rd_df.write.json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer",mode="Overwrite")

In [0]:
#2. Write usage data into JSON format using append mode and snappy compression format
wr_json2=rd_df1.write.json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/usage",mode="append")#without compression format, its providing result in readable json format
wr_json2=rd_df1.write.json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/usage",mode="append",compression="snappy") #with snappy compression format

In [0]:
#3. Write tower data into JSON format using ignore mode and observe the behavior of this mode
wr_json3=rd_df2.write.json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/tower/",mode="append")#tried ignore > it does not create any impact, for further testing added mode as append

In [0]:
#4. Read the tower data in a dataframe and show only 5 rows.
read_json_df=spark.read.json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/tower/*.json")
display(read_json_df.limit(5))

##8. Write Operations (Data Conversion/Schema migration) – Parquet Format Usecases
1. Write customer data into Parquet format using overwrite mode and in a gzip format
2. Write usage data into Parquet format using error mode
3. Write tower data into Parquet format with gzip compression option
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:
#read data
rd_df=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",inferSchema=True).toDF("custid","name","age","location","plan")

#1. Write customer data into Parquet format using overwrite mode and in a gzip format
rd_df.write.option("compression","gzip").parquet(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer_parquet/",mode="overwrite")

dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer_parquet/")

#2. Write usage data into Parquet format using error mode
rd_df5=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage_updated.csv",header=True,inferSchema=True)
rd_df5.write.parquet(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/usage_parquet/",mode="overwrite") #error

#3. gzip compression tried in steps-1

#4. Read the usage data in a dataframe and show only 5 rows.
spark.read.parquet("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/usage_parquet").limit(5).display()

##9. Write Operations (Data Conversion/Schema migration) – Orc Format Usecases
1. Write customer data into ORC format using overwrite mode
2. Write usage data into ORC format using append mode
3. Write tower data into ORC format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.

In [0]:
#read data
rd_df=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",inferSchema=True).toDF("custid","name","age","city","plan")

#ORC format using overwrite, append mode
#rd_df.write.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer_orc",mode="append")

rd_df.write.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer_orc",mode="overwrite")

#read orc format output
readorc=spark.read.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer_orc")
display(readorc.limit(5))

##10. Write Operations (Data Conversion/Schema migration) – Delta Format Usecases
1. Write customer data into Delta format using overwrite mode
2. Write usage data into Delta format using append mode
3. Write tower data into Delta format and see the output file structure
4. Read the usage data in a dataframe and show only 5 rows.
5. Download the file into local harddisk from the catalog volume location and see the data of any of the above files opening in a notepad++.
6. Compare the parquet location and delta location and try to understand what is the differentiating factor, as both are parquet files only.

In [0]:
#read data
rd_df=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",inferSchema=True).toDF("custid","name","age","city","plan")

#1. Write customer data into Delta format using overwrite mode
target_delta="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer_delta"
rd_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").save(target_delta)

#read delta format data
spark.read.format("delta").load(target_delta).show(5)

##11. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using saveAsTable() as a managed table
2. Write usage data using saveAsTable() with overwrite mode
3. Drop the managed table and verify data removal
4. Go and check the table overview and realize it is in delta format in the Catalog.
5. Use spark.read.sql to write some simple queries on the above tables created.


In [0]:
#read cust data
rd_df=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",inferSchema=True).toDF("custid","name","age","city","plan")

#1. Write customer data using saveAsTable() as a managed table
rd_df.write.saveAsTable("telecom_catalog_assign.landing_zone.cust_tb1",mode="overwrite")
#Type: Managed
#Data source: Delta

#2. show create table
display(
    #spark.sql("show create table telecom_catalog_assign.landing_zone.cust_tb1")
    spark.sql("select * from telecom_catalog_assign.landing_zone.cust_tb1")
)

#3. Drop the managed table
spark.sql("drop table telecom_catalog_assign.landing_zone.cust_tb1")

##12. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data using insertInto() in a new table and find the behavior
2. Write usage data using insertTable() with overwrite mode

##13. Write Operations (Lakehouse Usecases) – Delta table Usecases
1. Write customer data into XML format using rowTag as cust
2. Write usage data into XML format using overwrite mode with the rowTag as usage
3. Download the xml data and open the file in notepad++ and see how the xml file looks like.

In [0]:
#read cust data
rd_df=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",inferSchema=True).toDF("custid","name","age","city","plan")

rd_df.write.xml("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer_xml",mode="ignore",rowTag="cust")

#read cust data in xml format
read_xmp=spark.read.xml("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/target_zone/customer_xml",rowTag="cust")
display(read_xmp)

##14. Compare all the downloaded files (csv, json, orc, parquet, delta and xml) 
1. Capture the size occupied between all of these file formats and list the formats below based on the order of size from small to big.

## 15. Try to do permutation and combination of performing Schema Migration & Data Conversion operations like...
1. Read any one of the above orc data in a dataframe and write it to dbfs in a parquet format
2. Read any one of the above parquet data in a dataframe and write it to dbfs in a delta format
3. Read any one of the above delta data in a dataframe and write it to dbfs in a xml format
4. Read any one of the above delta table in a dataframe and write it to dbfs in a json format
5. Read any one of the above delta table in a dataframe and write it to another table

##16. Do a final exercise of defining one/two liner of... 
1. When to use/benifits csv
2. When to use/benifits json
3. When to use/benifit orc
4. When to use/benifit parquet
5. When to use/benifit delta
6. When to use/benifit xml
7. When to use/benifit delta tables
